In [1]:
import pandas as pd
df = pd.read_csv("./metadata.csv",header=None,names=["journal","title","doi","link"])

df

,journal,title,doi,link
0,Vis,Interdisciplinary visualization: lessons learn...,10.0000/00000002,http://dl.acm.org/citation.cfm?id=949606&CFID=...
1,Vis,Surface representations of two- and three-dime...,10.1109/VISUAL.1990.146359,http://dx.doi.org/10.1109/VISUAL.1990.146359
2,Vis,FAST: a multi-processed environment for visual...,10.1109/VISUAL.1990.146360,http://dx.doi.org/10.1109/VISUAL.1990.146360
3,Vis,The VIS-5D system for easy interactive visuali...,10.1109/VISUAL.1990.146361,http://dx.doi.org/10.1109/VISUAL.1990.146361
4,Vis,A procedural interface for volume rendering,10.1109/VISUAL.1990.146362,http://dx.doi.org/10.1109/VISUAL.1990.146362
...,...,...,...,...
3096,VAST,RegressionExplorer: Interactive Exploration of...,10.1109/TVCG.2018.2865043,http://dx.doi.org/10.1109/TVCG.2018.2865043
3097,VAST,Seq2Seq-Vis: A Visual Debugging Tool for Seque...,10.1109/TVCG.2018.2865044,http://dx.doi.org/10.1109/TVCG.2018.2865044
3098,VAST,"SIRIUS: Dual, Symmetric, Interactive Dimension...",10.1109/TVCG.2018.2865047,http://dx.doi.org/10.1109/TVCG.2018.2865047
3099,VAST,MotionRugs: Visualizing Collective Trends in S...,10.1109/TVCG.2018.2865049,http://dx.doi.org/10.1109/TVCG.2018.2865049


In [2]:
dois = df[df.link.str.contains("doi.org")]

In [3]:
dois

,journal,title,doi,link
1,Vis,Surface representations of two- and three-dime...,10.1109/VISUAL.1990.146359,http://dx.doi.org/10.1109/VISUAL.1990.146359
2,Vis,FAST: a multi-processed environment for visual...,10.1109/VISUAL.1990.146360,http://dx.doi.org/10.1109/VISUAL.1990.146360
3,Vis,The VIS-5D system for easy interactive visuali...,10.1109/VISUAL.1990.146361,http://dx.doi.org/10.1109/VISUAL.1990.146361
4,Vis,A procedural interface for volume rendering,10.1109/VISUAL.1990.146362,http://dx.doi.org/10.1109/VISUAL.1990.146362
5,Vis,Techniques for the interactive visualization o...,10.1109/VISUAL.1990.146363,http://dx.doi.org/10.1109/VISUAL.1990.146363
...,...,...,...,...
3096,VAST,RegressionExplorer: Interactive Exploration of...,10.1109/TVCG.2018.2865043,http://dx.doi.org/10.1109/TVCG.2018.2865043
3097,VAST,Seq2Seq-Vis: A Visual Debugging Tool for Seque...,10.1109/TVCG.2018.2865044,http://dx.doi.org/10.1109/TVCG.2018.2865044
3098,VAST,"SIRIUS: Dual, Symmetric, Interactive Dimension...",10.1109/TVCG.2018.2865047,http://dx.doi.org/10.1109/TVCG.2018.2865047
3099,VAST,MotionRugs: Visualizing Collective Trends in S...,10.1109/TVCG.2018.2865049,http://dx.doi.org/10.1109/TVCG.2018.2865049


In [4]:
# %load ../../vis-sieve/openalex_code/database/create_database.py
import duckdb as db

def create(name):
    # TODO add in code to handle if the .db isn't added
    con = db.connect(f'{name}')
    con.execute('''
                DROP TABLE IF EXISTS residence;
                DROP TABLE IF EXISTS contribution;
                DROP TABLE IF EXISTS figure_property;
                DROP TABLE IF EXISTS figure;
                DROP TABLE IF EXISTS author;
                DROP TABLE IF EXISTS institution;
                DROP TABLE IF EXISTS paper;
                ''')


    con.execute('''
    CREATE TABLE author (
        id BIGINT,
        name VARCHAR(100) NOT NULL
    );
                
    CREATE TABLE institution (
        id BIGINT ,
        ror VARCHAR(20) NOT NULL,
        name VARCHAR(100) NOT NULL
    );
                
    CREATE TABLE residence (
        au_id BIGINT,
        inst_id BIGINT,
    );
                
    CREATE TABLE paper (
        id BIGINT ,
        title VARCHAR(200) NOT NULL,
        doi VARCHAR(100),
        publication_date DATE,
        oa_url VARCHAR(200),
        pdf_path VARCHAR(150),
        inst_id BIGINT,
    );
                
    CREATE TABLE contribution (
        au_id BIGINT,
        paper_id BIGINT,
    );
                
    CREATE TABLE figure (
        id BIGINT ,
        paper_id BIGINT,
        local_path VARCHAR(150),
        server_path VARCHAR(150),
    );
                
    CREATE TABLE figure_property (
        name VARCHAR(100),
        int_value INTEGER,
        string_value VARCHAR(100),
        figure_id BIGINT,
    );
                

    ''')

    con.close()

In [20]:
# %load ../../vis-sieve/openalex_code/hear_me_ROR_script.py
"""
file: hear_me_ROR_script.py
author: Ben Kruse
Adapted from: hear_me_ROR.ipynb by Devin Bayly

Takes a ROR identification of school and a range of years,
then generates a json file with the publications of the school
for that period
"""

# from database.create_database import create
from pathlib import Path
import os
import requests as rq
import json
import argparse
from tqdm import tqdm
import math
import duckdb as db

def remove_duplicate_authors(publications, silent=False):
    """ Removes duplicate authors from a list of publications

    Args:
        publications (list): list of publications to remove duplicate authors from

    Returns:
        list: list of publications with duplicate authors removed
    """
    for pub in publications:
        authors = pub["authorships"]
        seen_ids = set()
        new_authors = []
        for a in authors:
            if a["author"]["id"] not in seen_ids:
                new_authors.append(a)
                seen_ids.add(a["author"]["id"])
            else: 
                print(f"Duplicate author in {pub['title']}: {a['author']['display_name']}")
        pub["authorships"] = new_authors
    return publications

def results_per_year(year, ror="03m2x1q45", silent=False, filter_duplicate_authors=True, testing=False):
    """ Gets the publications for a school for a year

    Args:
        year (int): year to get publications for
        ror (str): ROR identification of the school

    Returns:
        list: list of publications for the school for the year
    """
    all_res = []
    headers = {"mailto":"baylyd@arizona.edu"} ## Make command line arg
    res = rq.get(f"https://api.openalex.org/works?filter=publication_year:{year},institutions.ror:{ror}&cursor=*&per-page=200",headers=headers)
    data = res.json()
    if filter_duplicate_authors:
        data["results"] = remove_duplicate_authors(data["results"], silent=silent)
    page_count = data["meta"]["count"]/data["meta"]["per_page"]
    all_res.extend(data["results"])
    cursor = data["meta"]["next_cursor"]
    query = 0
    if not silent:
        pbar = tqdm(total=math.ceil(page_count))
    while cursor:
        query+=1
        res = rq.get(f"https://api.openalex.org/works?filter=publication_year:{year},institutions.ror:{ror}&cursor={cursor}&per-page=200")
        
        # Sometimes request fails, leave the year (find better way)
        try:
            data = res.json()
        except json.decoder.JSONDecodeError:
            print(f"Error on query {query}")
            print(res.text)
            break

        if filter_duplicate_authors:
            data["results"] = remove_duplicate_authors(data["results"], silent=silent)
        all_res.extend(data["results"])
        cursor = data["meta"].get("next_cursor",None)
        if not silent:
            pbar.update(1)
        if testing:
            break
    return all_res
def doi_result(doi_url,testing=False):
    """ Gets the publications for a school for a year

    Args:
        year (int): year to get publications for
        ror (str): ROR identification of the school

    Returns:
        list: list of publications for the school for the year
    """
    all_res = []
    headers = {"mailto":"baylyd@arizona.edu"} ## Make command line arg
    res = rq.get(f"https://api.openalex.org/works/{doi_url}",headers=headers)
    data = res.json()
    if testing:
        print(data)
    
    return data

def get_publications(ror: str, years: range, output_file: str, silent=False, get_authors=False):
    """ Gets the publications for a school for a range of years and 
    writes them to a json file

    Args:
        ror (str): ROR identification of the school
        years (range): range of years to get publications for
    
    Returns:
        None
    """
    all_res = []
    for year in years:
        if not silent:
            print(f"Getting publications for {year}")
        all_res.extend(results_per_year(year, ror, silent))
    with open(output_file,"w") as f:
        json.dump(all_res,f)

    if get_authors:
        if not silent:
            print("Getting authors")
        with open("authors"+output_file,"w") as f:
            json.dump(get_all_authors(all_res, output_file), f)
        

def get_all_authors(publications, output_file):
    """ Gets the authors from a list of publications

    Args:
        publications (list): list of publications to get authors from

    Returns:
        list: list of authors from the publications
    """
    authors = {}
    author_ids = {}
    for pub in publications:
        for a in pub["authorships"]:
            publication = {"work_id": pub["id"], 
                           "author": {"id": a["author"]["id"], 
                                      "display_name": a["author"]["display_name"], 
                                      "orcid": a["author"]["orcid"]
                                      },
                            "open_access": pub["open_access"]
                            }
            if a["author"]["id"] not in author_ids:
                pub_array = [publication]
                author_ids[a["author"]["id"]] = pub_array
                authors[a["author"]["display_name"]] = pub_array
            else:
                author_ids[a["author"]["id"]].append(publication)

    return authors


def add_institution_to_db(con: db.DuckDBPyConnection, institution_ror: str = None, 
                          institution_name: str = None, institution_id: int = None) -> int:
    """ Adds an institution to the database

    Args:
        con (db.DuckDBPyConnection): connection to the database
        institution_ror (str): ROR identification of the institution

    Returns:
        int: id of the institution
    """

    if institution_ror is not None:
        institution_info = rq.get(f"https://api.openalex.org/institutions/https://ror.org/{institution_ror}").json()
        institution_name = institution_name or institution_info["display_name"]
        institution_id = institution_id or int(institution_info["id"].split("I")[-1])

    elif institution_id is not None:
        con.execute(f"SELECT COUNT(1) FROM institution WHERE id = {institution_id};")
        exists = con.fetchone()[0]
        if exists:
            pass
            return institution_id
        
        institution_info = rq.get(f"https://api.openalex.org/institutions/I{institution_id}").json()
        institution_name = institution_name or institution_info["display_name"]
        institution_ror = institution_ror or institution_info["ror"].split("/")[-1]
        
    institution_name = institution_name.replace("'", "")
    try:
        con.execute(f"INSERT INTO institution VALUES ({institution_id}, '{institution_ror}', '{institution_name}');")
    except db.ConstraintException:
        pass

    return institution_id
    
def populate_database(database_file: str, ror: str, years: range, content_root: str,
                      json_output: str = None, silent: bool = False,dois=None) -> None:
    """ Populates a database with publications for a school for a range of years

    Args:
        database_file (str): name of the database file to populate
        ror (str): ROR identification of the school
        years (range): range of years to get publications for
        json_output (str): name of the json file to write the publications to
        silent (bool): silence output
    """
    con = db.connect(database_file)
    inst_id = add_institution_to_db(con, institution_ror=ror)
    failed_rows =[]
    all_results =[]
    if type(dois) !=type(None):
        # iterate over the dois, and make a similar function as results_per_year but to retrieve a single result provided a doi string
        for index,row in tqdm(dois.iterrows()):
            
            link = row.link
            pub = doi_result(link)
            # double check that we are getting a dictionary back not a list of results
            
            if type(pub) == type(dict()):
                all_results.append(pub)
                # this is a list of probably one element or more if multiple matched the doi
            
            
                # Add publication to database
                #await add_publication_and_figures(con, pub, content_root, playwright)
                # TODO wrap these publication table lines in a function
                
                pub_date = pub["publication_date"]
                pub_id = int(pub["id"].split("W")[-1])
                title = pub["title"]
                if not title:
                  print("title is none ",pub)
                  continue
                pub_title = title[:200].replace("'", "")
                pub_doi = pub["doi"]
                pub_oa_url = pub["open_access"]["oa_url"]
                pub_inst_id = inst_id
                pub_oa_status = pub["open_access"]["oa_status"]
                # # this section is checking for whether the table has the paper in it already
                # con.execute(f"SELECT COUNT(1) FROM paper WHERE id = {pub_id};")
                # exists = con.fetchone()[0]
                
                # if exists:
                #     # want to continue to the next publication and not try to update either paper table, or the authorship tables, because this has already happened
                #     continue
    
                con.execute(f"""INSERT INTO paper (id, title, doi, publication_date, oa_url,inst_id) VALUES ({pub_id}, '{pub_title}', '{pub_doi}', '{pub_date}', '{pub_oa_url}','{pub_inst_id}');""")
                for a in pub["authorships"]:
                    # TODO make sure filter the authors and only include the people that are actually affiliated with our ROR code 
                    institutions = a["institutions"]
                    # get the rors from the institutions author is affiliated with
                    # use path to trim off only the last part
                    rors = [Path(i["ror"]).stem for i in institutions]
                    # only add the author to the table if we see their affiliation with the university 
                    try:
                        con.execute(f"""INSERT INTO author VALUES ({a['author']['id'].split('A')[-1]}, '{a['author']['display_name'].replace("'", "")}');""")
                    except db.ConstraintException:
                        pass

                    try:
                        con.execute(f"INSERT INTO contribution VALUES ({a['author']['id'].split('A')[-1]}, {pub['id'].split('W')[-1]});")
                    except db.ConstraintException:
                        pass
                    
                    for author_inst in a["institutions"]:
                        # Making another change from Ben's code
                        add_institution_to_db(con, institution_id=int(author_inst["id"].split("I")[-1]))
                        try:
                            con.execute(f"INSERT INTO residence VALUES ({a['author']['id'].split('A')[-1]}, {author_inst['id'].split('I')[-1]});")
                        except db.ConstraintException:
                            pass
            else:
                failed_rows.append(row)
    Path(json_output).write_text(json.dumps(all_results))
    return failed_rows,all_results

In [5]:
from pathlib import Path

In [25]:
# recreating setup
db_file = Path("visimages.db")
create(db_file)
fails,successes = populate_database(db_file, "03m2x1q45", range(2000,2025),"./openalex_visimages", "Hear_me_ROR_out_visimages.json", True,dois)
# get all authors

3053it [20:00,  2.54it/s]


In [26]:
len(successes)

3053

In [21]:
all_authors = get_all_authors(successes,"./Hear_me_ROR_out_visimages.json")

In [5]:
import duckdb as db

In [6]:
con = db.connect("visimages.db")

In [7]:
con.sql("SHOW TABLES").show()

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ author          │
│ contribution    │
│ figure          │
│ figure_property │
│ institution     │
│ paper           │
│ residence       │
└─────────────────┘



In [8]:
con.sql("SELECT * FROM PAPER")

┌────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────┬──────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬──────────┬───────────┐
│     id     │                                                         title                                                          │                    doi                     │ publication_date │                                        oa_url                                         │ pdf_path │  inst_id  │
│   int64    │                                                        varchar                                                         │                  varchar                   │       date       │                                        varchar                                        │ varchar  │   int64   │
├────────────┼─────────────────────────────────────────────────────

In [9]:
type(None) != type(None)

False

In [10]:
[row.link for index,row in dois.iterrows()]

['http://dx.doi.org/10.1109/VISUAL.1990.146359',
 'http://dx.doi.org/10.1109/VISUAL.1990.146360',
 'http://dx.doi.org/10.1109/VISUAL.1990.146361',
 'http://dx.doi.org/10.1109/VISUAL.1990.146362',
 'http://dx.doi.org/10.1109/VISUAL.1990.146363',
 'http://dx.doi.org/10.1109/VISUAL.1990.146364',
 'http://dx.doi.org/10.1109/VISUAL.1990.146365',
 'http://dx.doi.org/10.1109/VISUAL.1990.146366',
 'http://dx.doi.org/10.1109/VISUAL.1990.146367',
 'http://dx.doi.org/10.1109/VISUAL.1990.146368',
 'http://dx.doi.org/10.1109/VISUAL.1990.146369',
 'http://dx.doi.org/10.1109/VISUAL.1990.146370',
 'http://dx.doi.org/10.1109/VISUAL.1990.146371',
 'http://dx.doi.org/10.1109/VISUAL.1990.146372',
 'http://dx.doi.org/10.1109/VISUAL.1990.146373',
 'http://dx.doi.org/10.1109/VISUAL.1990.146374',
 'http://dx.doi.org/10.1109/VISUAL.1990.146375',
 'http://dx.doi.org/10.1109/VISUAL.1990.146376',
 'http://dx.doi.org/10.1109/VISUAL.1990.146377',
 'http://dx.doi.org/10.1109/VISUAL.1990.146378',
 'http://dx.doi.org/

In [11]:
con.sql("SELECT * FROM author")

┌────────────┬──────────────────────┐
│     id     │         name         │
│   int64    │       varchar        │
├────────────┼──────────────────────┤
│ 5071520966 │ James L. Helman      │
│ 5110998886 │ L. Hesselink         │
│ 5006450303 │ Gordon Bancroft      │
│ 5083931343 │ F. Merritt           │
│ 5030050956 │ Todd Plessel         │
│ 5089904045 │ Paul G. Kelaita      │
│ 5067876179 │ R. Kevin McCabe      │
│ 5073465792 │ Al Globus            │
│ 5046609142 │ B. Hibbard           │
│ 5070589478 │ David Santek         │
│      ·     │      ·               │
│      ·     │      ·               │
│      ·     │      ·               │
│ 5010847113 │ Po-Ming Law          │
│ 5051526027 │ Wenchao Wu           │
│ 5108098662 │ Yixian Zheng         │
│ 5091466289 │ Huamin Qu            │
│ 5012158528 │ Sriram Karthik Badam │
│ 5061444760 │ Fereshteh Amini      │
│ 5034277315 │ Niklas Elmqvist      │
│ 5078907679 │ Pourang Irani        │
│ 5061027107 │ Florian Heimerl      │
│ 5112771943

In [12]:
con.sql("SELECT * FROM paper")

┌────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────┬──────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬──────────┬───────────┐
│     id     │                                                         title                                                          │                    doi                     │ publication_date │                                        oa_url                                         │ pdf_path │  inst_id  │
│   int64    │                                                        varchar                                                         │                  varchar                   │       date       │                                        varchar                                        │ varchar  │   int64   │
├────────────┼─────────────────────────────────────────────────────

In [13]:
con.sql("SELECT * FROM institution").show()

┌────────────┬───────────┬──────────────────────────────────────────────┐
│     id     │    ror    │                     name                     │
│   int64    │  varchar  │                   varchar                    │
├────────────┼───────────┼──────────────────────────────────────────────┤
│  138006243 │ 03m2x1q45 │ University of Arizona                        │
│   97018004 │ 00f54p054 │ Stanford University                          │
│   13805885 │ 056e22e24 │ Vaughn College of Aeronautics and Technology │
│ 4210153694 │ 049ba1e57 │ Rho (United States)                          │
│  135310074 │ 01y2jtd41 │ University of Wisconsin–Madison              │
│  126533617 │ 04k8zab17 │ Alliant International University             │
│   55732556 │ 03efmqc40 │ Arizona State University                     │
│   66946132 │ 047s2c258 │ University of Maryland, College Park         │
│   59553526 │ 05qghxh33 │ Stony Brook University                       │
│ 1327163397 │ 01q1z8k08 │ State Unive

In [31]:
con.close()

In [5]:
import duckdb as db

In [6]:
con = db.connect("./visimages.db")

In [33]:
con.execute("EXPORT DATABASE 'vis-parquet' (FORMAT parquet);")

In [7]:
papers = con.sql("SELECT * FROM paper").df()

In [10]:
papers.iloc[285]

id                                                         1970929543
title               Research report: information animation applica...
doi                        https://doi.org/10.1109/infvis.1995.528682
publication_date                                  2002-11-19 00:00:00
oa_url                                                           None
pdf_path                                                         None
inst_id                                                     138006243
Name: 285, dtype: object